# Exercise Sheet 07 - Solutions

This markdown block is used to define macros for later markdown blocks. 
$\newcommand{\normaldist}{\mathcal{N}}$
$\newcommand{\betadist}{\text{Beta}}$
$\newcommand{\reals}{\mathbb{R}}$
$\newcommand{\ML}{\text{ML}}$
$\renewcommand{\vec}[1]{\boldsymbol{\mathbf{#1}}}$
$\newcommand{\matrix}[1]{\boldsymbol{\mathbf{#1}}}$
$\newcommand{\dataset}{\mathcal{D}}$
$\newcommand{\class}{\mathcal{C}}$
$\newcommand{\optimal}[1]{#1^{\star}}$
$\newcommand{\argmin}{\text{argmin}}$
$\newcommand{\argmax}{\text{argmax}}$
$\newcommand{\expct}{\mathbb{E}}$
$\newcommand{\entropy}[1]{H[#1]}$
$\newcommand{\conditionalentropy}[2]{\entropy{#1 | #2}}$
$\newcommand{\kldiv}[2]{KL(#1 || #2)}$
$\newcommand{\mutualinfo}[2]{I[#1;#2]}$
$\newcommand{\deriv}[2]{\frac{d}{d #2} \left( #1\right)}$
$\newcommand{\inputs}{\matrix{X}}$
$\newcommand{\identitymtx}{\matrix{I}}$
$\newcommand{\designmtx}{\matrix{\Phi}}$
$\newcommand{\featurevec}{\boldsymbol{\phi}}$
$\newcommand{\norm}[1]{\lVert #1 \rVert}$
$\newcommand{\grad}{\nabla}$

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import scipy.stats
from itertools import groupby

## Exercise 7.0

Download the Breast Cancer Wisconsin (Diagnostic) data from the UCI repository [here](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) and copy the `wdbc.data` file to the same folder as this notebook. You will need to extract the zip file and locate the data file to do this. This data gives summarised statistics of image features of a microscopic image of cells drawn during a breast cancer biopsy.

### 💻 7.0 a)

Complete the first code block below to load the data and extract to numpy arrays. You will need to have copied the data as described above. 

The code uses the `read_csv` file to read in the data. Note that by default `read_csv` uses the first line of the data file as the header, which would provide the column names for the data-frame. However, our data does not have a header line. Instead, we set `header=None` to treat the first line as data, and provide the column names separately with the `names` argument. Can you see which parts of the code this refers to?

Finally, the input data and targets are extracted to variables `X` and `y` respectively. However, there is a problem, the targets are of type `str` with `"M"` for malignant (the positive class, target `1`) and `"B"` for benign (the negative class, target `0`). Use the `map` function to convert these targets to the correct type.

**Hint:** Read [the documentation for the function `map`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.map.html) to see how to convert the values. You can also read [the documentation for `read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) to understand how this function is used.

### 💻 7.0 b)

When **Ex. 7.0 a)** is complete run the second code block below. This fits a linear model to the data (as we saw in our Units on regression), by minimising the squared error between the prediction and the targets. Given that each target is `0` or `1` can you see a problem with this. Does a low RMSE on the test data correspond to a high accuracy?

### 💻 7.0 c)

The third code block below aims to calculate the accuracy of the model predictions. Part of this has been done for you, the real valued predictions from the model have been extracted to `y_pred_reals`. You should convert these to `0-1` binary predictions using the following simple rule:
* if the real prediction is non-negative, set the binary prediction to `1`
* if the real prediction is negative, set the binary prediction to `0`

The final two lines of the code compares the binary predictions with the targets. 

* Can you see how this is done?
* Are you suprised by the result?

**Hint:** If you compare two equally sized numpy arrays with Python comparison operators: `>`, `<`, `<=`, `>=`, `==`  or `!=`, then the result is a numpy array of the same size with data-type (`dtype`) `bool`. This means the elements are `True` or `False`. You can convert this to an integer array by using the `astype` function with argument `dtype=int`.

**Note:** The fourth code block below, simply shows an alternative way to calculate the accuracy from the predictions using a built in `sklearn` function.

In [ ]:
# Ex 7.0 a) Load the data and extract to numpy arrays.

# Load the data, define and set the column names
# and display the resulting dataframe.
filename = "wdbc.data"
root_names = [
    "radius", "texture", "perimeter", "area",
    "smoothness", "compactness", "concavity",
    "concave points", "symmetry ", "fractal dimension"]
# There are three groups of features: means, standard errors (SE)
# and mean of largest three values (worst).
feature_names = [
    f"{measure} {root}"  for measure in ["mean", "SE", "worst"] for root in root_names]
target_name = "Diagnosis"
all_names = ["ID number", target_name] + feature_names
df = pd.read_csv(filename, index_col=0, header=None, names=all_names)
display(df)

# Extract the data/design matrix.
target_map = {"M": 1, "B":0}
X = df[feature_names].to_numpy()
# Extract targets and convert to binary.
y = df[target_name].map(target_map).to_numpy()
print("\nThe first five rows and columns of the data:")
display(X[:5,:5])
print("\nThe first five targets:")
display(y[:5])

In [ ]:
# 7.0 b) Split the data and train as a regression problem.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2)

# Train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Calculate Mean Squared Error (MSE) for train and test sets
train_error = root_mean_squared_error(y_train, model.predict(X_train))
test_error = root_mean_squared_error(y_test, model.predict(X_test))

# Report RMSE loss.
print(f'Train Error (RMSE): {train_error:.2f}')
print(f'Test Error (RMSE): {test_error:.2f}')

In [ ]:
# 7.0 c) Calculate the accuracy using Numpy arithmetic.

# Convert the positive or negative predictions into binary predictions.
y_pred_reals = model.predict(X_test)
y_pred_binary = (y_pred_reals >= 0).astype(int)

# Calculate the accuracy. How many binary predictions match the targets.
accuracy = np.mean(y_pred_binary == y_test)
print(f"accuracy = {accuracy:.3f}")

In [ ]:
# 7.0 d) An alternative accuracy score provided by sklearn.
accuracy = accuracy_score(y_test, y_pred_binary, normalize=True)
print(f"accuracy = {accuracy:.3f}")

## 💡Exercise 7.1

Go back to the lecture slides for Unit 5 and look at slide **A Simple Neuron** (slide number 5). How would this picture of a synthetic neuron have to change to accomodate the new model described in Result (7.1).

## 💡 Exercise 7.2

If you haven't already, then go back and re-read Section 1.5 in (Bishop, 2006). 

Which of the options from the slide **Probabilistic discriminative models can still...** would make sense in the context of the Cancer classification dataset from Unit 2?

## ✏️ Exercise 7.3

Think about the computational complexity with respect to different predictors. Imagine that your input is $M$ dimensional and you have $K$ classes that you wish to classify.

i. If you were to perform one-versus-many classification with linear models as discussed in the previous video, how many parameters would you need in total for all the binary discriminants?

ii. What about if you used the one-versus-one scheme?

iii. Finally, how many parameters would you need for the single $K$-class discriminant described on slide **A better discriminant for K -classes**.

## Exercise 7.4

For this question, you should also download the iris dataset from the moodle webpage. The original source of the data is the bezdekIris.data from [here](http://archive.ics.uci.edu/ml/datasets/iris).

### 💻 7.4 a)
Download the data to the same folder as this notebook and run the code-block below. This will import the data and generate an array of scatter-plots and histograms, such that each class is drawn in a different color. Notice that there are $3$ classes, and that *Iris-setosa* (purple) is obviously linearly separable from the other classes.

*You may be interested in the `import_for_classification` function in `./fomlads/data/external.py`. This uses the `pandas` library to import the data-file into a `DataFrame` object, so we can store different types of data in different columns. Documentation for the DataFrame object can be found [here](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.html). In this case, the class column contains string (text) values, and the other columns contain real numbers.*

### 💻 7.4 b)

Look at the `project_and_histogram_data` function in the second code block below. This function aims to project your data down onto 1 dimension, using a specified projection method, then histograms the resulting projected data, but it requires data with only 2 classes. The code reimports your data, but this time taking only classes `Iris-setosa` and `Iris-versicolor`.

The projection method implemented for you is the maximum mean separation method described in the lectures. This finds the projection vector, $\vec{w}$, that maximises the distance between projected class means. This projection is performed by the `maximum_separation_projection` function of `./fomlads/model/classification.py`. However, another function `project_data` in `./fomlads/model/classification.py` is not complete yet.

When you initially run the code block, this will produce a random histogram. To make this run properly, you will need to correct the function `project_data` in `./fomlads/model/classification.py` so that it returns a vector of 1d projections. Given a weight vector, $\vec{w}$, the projection for each input data-point, $\vec{x}$ (a row of `data`), is a single value equal to:
$$u = \vec{w}^T \vec{x}$$

*As in previous tutorials, there are ways of doing this using a for loop, and a better way of doing it as a single matrix operation.*

When completed it should be clear that your $2$ classes: Iris-setosa, Iris-virginica, are completely separated in the projected dimension. A classifier could therefore be constructed by choosing a threshold between the two histograms. We may have been able to anticipate this from the exploratory plots.

### ✏️ 7.4 c) [optional]

Show mathematically that the distance between projected means is maximised when the unit (length 1) weight vector is given by:
\begin{align}
\vec{w} 
& = \frac{(\vec{m}_1 - \vec{m}_0)}{\norm{\vec{m}_1 - \vec{m}_0}}
\notag
\end{align}
Note that this is an example of constrained optimisation with objective
\begin{align}
f(\vec{w}) = \vec{w}^T(\vec{m}_{1}-\vec{m}_{0})
\notag
\end{align}
and constraint $\norm{\vec{w}} = \sqrt{\vec{w}^T \vec{w}} = 1$. Read (Bishop, 2006, Apx. E) for a suggestion of how to proceed here. 

### 💻 7.4 d)

In the third code block below, the projection and histogram function is again invoked but with a different pair of classes. This should show that the histograms of the projected values for these two classes are not separated. It may be possible to separate the classes more effectively with a better projection though. To do this, you will implement a new function `fisher_linear_discriminant_projection` (in `./fomlads/model/classification.py`). Your new function should take two arguments: an input data matrix,  `inputs`; and a vector of target values, `targets`, and return a weight vector corresponding to Fisher's linear discriminant projection vector. 

For `fisher_linear_discriminant_projection` you will need to calculate the projection vector using Results (7.2) and (7.3). For the code to run properly you will need to import the newly defined `fisher_linear_discriminant_projection` for use within `project_and_histogram_data`.


*You can save yourself a bit of effort by noticing that the two components of $\vec{S}_{W}$ are closely related to the maximum-likelihood covariance matrices of the two classes considered separately, and making use of the provided function `max_lik_mv_gaussian` in `./fomlads.model.density_estimation.py`.*


In [ ]:
## provided code for Ex. 7.4 a)
from fomlads.data.external import import_for_classification

from fomlads.model.classification import project_data
from fomlads.model.classification import maximum_separation_projection

from fomlads.plot.exploratory import plot_scatter_array_classes
from fomlads.plot.exploratory import plot_class_histograms

ifname = 'iris.data'
inputs, targets, field_names, classes = import_for_classification(
    ifname)
plot_scatter_array_classes(
    inputs, targets, field_names=field_names, classes=classes)


In [ ]:
## provided code for Ex. 7.4 b)
def project_and_histogram_data(
        inputs, targets, method, title=None, classes=None):
    """
    Takes input and target data for classification and projects
    this down onto 1 dimension according to the given method,
    then histograms the projected data.

    parameters
    ----------
    inputs - a 2d input matrix (array-like), each row is a data-point
    targets - 1d target vector (array-like) -- can be at most 2 classes ids
        0 and 1
    """
    weights = get_projection_weights(inputs, targets, method)
    projected_inputs = project_data(inputs, weights)
    ax = plot_class_histograms(projected_inputs, targets)
    # label x axis
    ax.set_xlabel(r"$\mathbf{w}^T\mathbf{x}$")
    ax.set_title("Projected Data: %s" % method)
    if not classes is None:
        ax.legend(classes)
        
def get_projection_weights(inputs, targets, method):
    """
    Helper function for project_and_histogram_data
    Takes input and target data for classification and projects
    this down onto 1 dimension according to the given method

    parameters
    ----------
    inputs - a 2d input matrix (array-like), each row is a data-point
    targets - 1d target vector (array-like) -- can be at most 2 classes ids
        0 and 1
    returns
    -------
    weights - the projection vector
    """
    if len(np.unique(targets)) > 2:
        raise ValueError("This method only supports data with two classes")
    if method == 'maximum_separation':
        weights = maximum_separation_projection(inputs, targets)
    elif method == 'fisher':
        weights = fisher_linear_discriminant_projection(inputs, targets)
    else:
        raise ValueError("Unrecognised projection method")
    return weights


# reimport data, just 2 classes
classes = ["Iris-setosa", "Iris-versicolor"]
inputs, targets, field_names, classes = import_for_classification(
    ifname, classes=classes)
# now project the data and histogram the projections
project_and_histogram_data(inputs, targets, method='maximum_separation')

In [ ]:
### provided code for Ex. 7.4 d)

# reimport data, a different 2  classes
classes = ["Iris-virginica", "Iris-versicolor"]
inputs, targets, field_names, classes = import_for_classification(
    ifname, classes=classes)
# now project the data and histogram the projections
project_and_histogram_data(inputs, targets, method='maximum_separation')

## Exercise 7.5

### 💡 7.5 a)

How would you evaluate the effectiveness of your two projections from **Ex. 7.4**, given that you do not know in advance what your loss matrix might be for misclassifications?

You may want to look try plotting a receiver-operator characteristic (ROC) curve. There is a plotting function `plot_roc` in module `./fomlads/plot/evaluations.py`, which will plot the curve given the right inputs. Compare Fisher's projection vector with the maximum mean separation by plotting both on the same ROC curve. Which method is better for this data?

*If you want more detail on the ROC curve and how it is used, there is an excellent description on Wikipedia (see [here](https://en.wikipedia.org/wiki/Receiver_operating_characteristic)}), you could also try some of the online videos or look at (the first half of) \cite{Fawcett2006}.*


### 💻 7.5 b)


i. Choose the best threshold for Fisher's method using the accuracy? Indicate this classifier on your ROC curve.

ii. Choose the best threshold for Fisher's method using the F1-score? Indicate this classifier on your ROC curve.

iii. Explain your finding for these resuts. What other measures could be used to choose a best classifier?

In [ ]:
### Provided code for Ex. 7.5 a)
from fomlads.plot.evaluations import plot_roc

def construct_and_plot_roc(
        inputs, targets, method='maximum_separation', **kwargs):
    """
    Takes input and target data for classification and projects
    this down onto 1 dimension according to the given method,
    then plots roc curve for the data.

    parameters
    ----------
    inputs - a 2d input matrix (array-like), each row is a data-point
    targets - 1d target vector (array-like) -- can be at most 2 classes ids
        0 and 1
    """
    weights = get_projection_weights(inputs, targets, method)
    projected_inputs = project_data(inputs, weights)
    new_ordering = np.argsort(projected_inputs)
    projected_inputs = projected_inputs[new_ordering]
    targets = np.copy(targets[new_ordering])
    N = targets.size
    num_neg = np.sum(1-targets)
    num_pos = np.sum(targets)
    false_positive_rates = np.empty(N)
    true_positive_rates = np.empty(N)
    for i in range(projected_inputs.size):
        false_positive_rates[i] = np.sum(1-targets[i:])/num_neg
        true_positive_rates[i] = np.sum(targets[i:])/num_pos
    fig, ax = plot_roc(
        false_positive_rates, true_positive_rates, label=method, **kwargs)
    return fig, ax


## 💻 Exercise 7.6

Just for fun, you can rerun your code on the turtle dataset from last tutorial, by copying it into this weeks folder and rerunning the above experiments.